# Kaggriculture | Shop & Pasture — Control

A compact, reproducible Kaggriculture submission package.

## Method

An independent control route with explicit terminal-bank, demand-alignment, guarded latent-pasture, and closeout decisions. It is kept separate from the Top-10 package so the two active submissions test different policies.

The notebook writes a single competition output: `submission.tar.gz` .

In [ ]:
# Exact public archive, repackaged with local validation.
import base64
import hashlib
import io
import tarfile
from pathlib import Path

ARCHIVE_B64 = (
    '...'
    'Fb/iV/yKX/Er/gvG/wUTrYpwAGgBAA=='
)
EXPECTED_SHA256 = "d245e14e4958c18fd721c104d7cb7115c4d51c77256fd6bbba025ade6662bbb4"
EXPECTED_MEMBERS = ['agents/e749a_niklita_consensus_network.py', 'agents/e750a_place_funding_repair.py', 'agents/e766a_universal_kenjo_medoid.py', 'agents/e773a_demand_aligned_pasture_network.py', 'agents/e774a_terminal_animal_frontier.py', 'agents/e775a_latent_pasture_activation.py', 'agents/e776a_engine_exact_latent_pasture.py', 'artifacts/e706_top10_tapes/episode_101408728_seat1.py', 'artifacts/e751_current_top10_tapes/episode_102192548_seat1.py', 'configs/server_environment_20260807.json', 'e776_pkg/__init__.py', 'e776_pkg/entry.py', 'main.py']

payload = base64.b64decode(ARCHIVE_B64)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
    assert archive.getnames() == EXPECTED_MEMBERS
    for member in archive.getmembers():
        data = archive.extractfile(member).read()
        if member.name.endswith(".py"):
            compile(data, member.name, "exec")

Path("submission.tar.gz").write_bytes(payload)
print("submission.tar.gz ready")
print("sha256:", EXPECTED_SHA256)
print("archive members:", len(EXPECTED_MEMBERS))
